In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
BASE_DIR = Path(os.getcwd()).resolve().parent
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
customers = pd.read_csv(DATA_RAW / "customers.csv")
cards = pd.read_csv(DATA_RAW / "cards.csv")
transactions = pd.read_csv(DATA_RAW / "transactions.csv")
transactions["transaction_time"] = pd.to_datetime(transactions["transaction_time"])

(2000, 15)
(2801, 5)
(44217, 13)


In [ ]:
print(f"Data Matrix Dimensions:\n- Customers: {customers.shape}\n- Cards: {cards.shape}\n- Transactions: {transactions.shape}")

customer_id       0
age               0
income            0
city              0
account_type      0
join_date         0
tenure_days       0
risk_score        0
is_active         0
avg_txn_amount    0
std_txn_amount    0
total_spend       0
txn_count         0
fraud_count       0
fraud_rate        0
dtype: int64
card_id         0
customer_id     0
card_type       0
credit_limit    0
status          0
dtype: int64
transaction_id        0
customer_id           0
amount                0
merchant              0
channel               0
device_type           0
transaction_time      0
location              0
is_fraud              0
fraud_score           0
hour                  0
day_of_week           0
distance_from_home    0
dtype: int64


### Missing Values and Structural Integrity Audits

In [ ]:
print("Missing values per collection:")
print(transactions.isnull().sum(), "\n")

print("Duplicate record counts:")
print(f"Transactions: {transactions.duplicated().sum()}")

Fraud Rate: 10.21%


### Exploratory Target Baseline Distribution Metrics

In [ ]:
baseline_fraud_rate = transactions["is_fraud"].mean() * 100
print(f"Global Base Fraud Scale: {baseline_fraud_rate:.2f}%")
print("\nTarget split arrays:")
print(transactions["is_fraud"].value_counts())


print("Aggregate cross-tabulations (Normal vs Fraud Spend Metrics):")
print(transactions.groupby("is_fraud")["amount"].describe())


print("Spatial-Financial Deviation Means:")
print(transactions.groupby("is_fraud")[["amount", "distance_from_home"]].mean())

is_fraud
0    39702
1     4515
Name: count, dtype: int64

### Primary Relational Feature Mapping & Merging

In [ ]:
# Build holistic master baseline matrix
master_df = transactions.merge(customers, on="customer_id", how="left")

# Extract categorical groupings from payment instrument history
card_metrics = (
    cards.groupby("customer_id")
    .agg(
        number_of_cards=("card_id", "count"),
        max_credit_limit=("credit_limit", "max"),
        avg_credit_limit=("credit_limit", "mean"),
        blocked_cards=("status", lambda x: (x == "Blocked").sum()),
        has_credit_card=("card_type", lambda x: int("Credit" in x.values)),
    )
    .reset_index()
)

master_df = master_df.merge(card_metrics, on="customer_id", how="left")

master_df.to_parquet(DATA_PROCESSED / "stg_master_prepared.parquet", index=False)
print("[SUCCESS] Staging merge finalized and compiled to disk.")

merchant
Cryptocurrency    40.104031
Education          7.867325
Electronics       11.144996
Entertainment      4.475636
Food Delivery      3.562620
Fuel               4.179352
Gaming            10.508563
Groceries          4.301994
Healthcare         6.354335
Restaurant         5.415550
Shopping Mall      7.958199
Travel            15.343317
Name: is_fraud, dtype: float64